# Resources

> 

In [ ]:
#| default_exp resources

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Set

import io, os, re, sys, json
from pathlib import Path

from mcp.server.fastmcp import FastMCP  # official MCP Python SDK (FastMCP 2.0+)
from mcp.server.fastmcp.resources import FunctionResource

from nbdev_mcp.utils.config import load_bookmarks
from nbdev_mcp.utils.paths import require_project, project_summary, nbs_dir, settings_dict, env_file

## Resources

In [ ]:
#| export
def resource_project_summary() -> str:
    """Resource: JSON summary of the current project."""
    p = require_project()
    return json.dumps(project_summary(p), indent=2)

In [ ]:
#| export
def resource_projects() -> str:
    """Resource: JSON of saved project bookmarks and NBDEV_PROJECTS env variable."""
    data = {'bookmarks': load_bookmarks(), 'NBDEV_PROJECTS': os.environ.get('NBDEV_PROJECTS', '')}
    return json.dumps(data, indent=2)

In [ ]:
#| export
def resource_tree() -> str:
    """Resource: JSON listing of notebooks in the current project (limited to 600 files)."""
    p = require_project()
    nbs = nbs_dir(p)
    files: List[str] = []
    if nbs.exists():
        for q in nbs.rglob('*.ipynb'):
            try:
                files.append(str(q.relative_to(p)))
            except Exception:
                files.append(str(q))
    payload = {'root': str(p), 'nbs_dir': str(nbs), 'notebooks': files[:600], 'has_settings_ini': (p / 'settings.ini').exists(), 'has_readme': (p / 'README.md').exists()}
    return json.dumps(payload, indent=2)

In [ ]:
#| export
def resource_settings() -> str:
    """Resource: contents of the current project's settings.ini file (text)."""
    p = require_project()
    f = p / 'settings.ini'
    return f.read_text(encoding='utf-8') if f.exists() else 'No settings.ini found.'

In [ ]:
#| export
def resource_env_file() -> str:
    """Resource: contents of the current project's environment YAML file (text)."""
    p = require_project()
    ef = env_file(p)
    return ef.read_text(encoding='utf-8') if ef.exists() else f'No env file at {ef}'


In [ ]:
#| export
def resource_read_file(relpath: str) -> str:
    """Resource: read a file by relative path within the current project (text content)."""
    p = require_project()
    f = (p / relpath).resolve()
    try:
        f.relative_to(p)
    except Exception:
        return f'Refusing to read outside project: {f}'
    if not f.exists():
        return f'Not found: {f}'
    try:
        return f.read_text(encoding='utf-8')
    except Exception as e:
        return f'Could not read {f}: {e}'

In [ ]:
#| export
def resource_index_to_readme_note() -> str:
    """Resource: JSON note explaining that nbs/index.ipynb becomes README.md in nbdev."""
    p = require_project()
    lib = settings_dict(p).get('lib_name') or '<your_lib>'
    msg = 'In nbdev, `nbs/index.ipynb` becomes `README.md` (the docs home page).'
    return json.dumps({'lib': lib, 'message': msg}, indent=2)

In [ ]:
#| export
def resource_learning_links() -> str:
    """Resource: curated links for nbdev, fastai, and related documentation.
    
    Returns a JSON object with categorized links to official documentation,
    tutorials, and community resources.
    """
    payload = {
        "nbdev": {
            "docs": "https://nbdev.fast.ai/",
            "getting_started": "https://nbdev.fast.ai/tutorials/tutorial.html",
            "best_practices": "https://nbdev.fast.ai/tutorials/best_practices.html",
            "directives": "https://nbdev.fast.ai/explanations/directives.html",
            "git_friendly": "https://nbdev.fast.ai/tutorials/git_friendly_jupyter.html",
            "github": "https://github.com/fastai/nbdev",
        },
        "fastai": {
            "docs": "https://docs.fast.ai/",
            "course": "https://course.fast.ai/",
            "github": "https://github.com/fastai/fastai",
            "forums": "https://forums.fast.ai/",
        },
        "fastcore": {
            "docs": "https://fastcore.fast.ai/",
            "github": "https://github.com/fastai/fastcore",
        },
        "execnb": {
            "docs": "https://execnb.fast.ai/",
            "github": "https://github.com/fastai/execnb",
        },
        "related": {
            "quarto": "https://quarto.org/",
            "jupyter": "https://jupyter.org/",
            "mcp_protocol": "https://modelcontextprotocol.io/",
        },
    }
    return json.dumps(payload, indent=2)

In [ ]:
#| export
def add_resources(mcp: FastMCP) -> None:
    """Register all nbdev-mcp resources with the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance to register resources with.
    """
    mcp.add_resource(FunctionResource(
        uri="nbdev://project", name="project_summary",
        description="JSON summary of the current nbdev project",
        fn=resource_project_summary
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://projects", name="projects",
        description="JSON of saved project bookmarks",
        fn=resource_projects
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://tree", name="tree",
        description="JSON listing of notebooks in the project",
        fn=resource_tree
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://settings", name="settings",
        description="Contents of settings.ini",
        fn=resource_settings
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://env", name="env_file",
        description="Contents of the environment YAML file",
        fn=resource_env_file
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://index-readme", name="index_to_readme",
        description="Note about nbs/index.ipynb becoming README.md",
        fn=resource_index_to_readme_note
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://links", name="learning_links",
        description="Curated links for nbdev/fastai documentation",
        fn=resource_learning_links
    ))

## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()